In [13]:
import sys
import site
site.addsitedir(site.getusersitepackages())
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sklearn
from scipy.io import loadmat
import sktime


In [4]:
## Directory of all .MAT files
matdata_dir = "./data/matfiles"

## Disposition
disposition = pd.read_csv("./data/disposition.csv")

## All analyses should only use files marked "Reduced"
reduced_disp = disposition[disposition['Reduced'] == "X"]

## Add .mat to DAQs so they can be read from the mat data directory, reset index to facilitate looping
reduced_disp = reduced_disp.assign(MatName = reduced_disp['DaqName'].str[:-4] + '.mat').reset_index(0, drop = True)

In [ ]:
## Empty final df
final_df = pd.DataFrame(columns=['Subject', 'Drive', 'LPupil_Diameter', 'RPupil_Diameter', 'Lateral_Deviation', 'Sample', 'Blink_Duration',
                                 'Left_Gaze_Dir_X', 'Left_Gaze_Dir_Y', 'Left_Gaze_Dir_Z', 'Right_Gaze_Dir_X', 'Right_Gaze_Dir_Y',
                                 'Right_Gaze_Dir_Z', 'Gaze_Pitch', 'Gaze_Yaw', 'Left_Eyelid_Closed', 'Right_Eyelid_Closed',
                                 'Vehicle_Speed', 'Brake_Pedal_Force', 'Steering_Wheel_Angle', 'Steering_Wheel_Angle_Rate', 'Accelerator_Pedal_Position',
                                 'Head_Pos_X', 'Head_Pos_Y', 'Head_Pos_Z', 'Blink_Counter', 'Blink_Frequency', 'Right_Button_Press', 'Left_Button_Press', 'DaqName'])
 
for i in range(0, reduced_disp.shape[0] - 1):
   
    ## Read current mat file
    fname = matdata_dir + "/" + reduced_disp['MatName'][i]
    try:
        data = loadmat(fname)
        print(f"Successfully read {fname} ({i+1}/{reduced_disp.shape[0]})")
    except FileNotFoundError:
        print(f"Skipping {fname}: File not found ({i+1}/{reduced_disp.shape[0]})")

   
    ## Vars to keep (just lane dev and pupil diam for now)
   
    ## Full lat pos series
    lat_pos_series = data['elemDataI']['SCC_Lane_Deviation'][0][0][:,1]
    lat_pos_series = lat_pos_series.flatten()
    nstep = lat_pos_series.shape[0]
   
    ## Full left pupil series
    try:
        left_pupil_series = data['elemDataI']['Output_FovioDMEResults_dme_core_pupil_left_pupil_diameter_mm'][0][0]
        left_pupil_series = left_pupil_series.flatten()
    except ValueError:
        left_pupil_series = np.repeat(99, nstep)
 
    ## Full right pupil series
    try:
        right_pupil_series = data['elemDataI']['Output_FovioDMEResults_dme_core_pupil_right_pupil_diameter_mm'][0][0]
        right_pupil_series = right_pupil_series.flatten()
    except ValueError:
        right_pupil_series = np.repeat(99, nstep)
 
    ## Blink counter series
    try:
        blink_counter_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_blink_counter'][0][0]
        blink_counter_series = blink_counter_series.flatten()
    except ValueError:
        blink_counter_series = np.repeat(99, nstep)
 
    ## Blink frequency series
    try:
        blink_frequency_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_blink_frequency'][0][0]
        blink_frequency_series = blink_frequency_series.flatten()
    except ValueError:
        blink_frequency_series = np.repeat(99, nstep)
   
    ## Head pos x series
    try:
        head_pos_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_x'][0][0]
        head_pos_x_series = head_pos_x_series.flatten()
    except ValueError:
        head_pos_x_series = np.repeat(99, nstep)
   
    ## Head pos y series
    try:
        head_pos_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_y'][0][0]
        head_pos_y_series = head_pos_y_series.flatten()
    except ValueError:
        head_pos_y_series = np.repeat(99, nstep)
 
    ## Head pos z series
    try:
        head_pos_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_head_head_pose_translation_z'][0][0]
        head_pos_z_series = head_pos_z_series.flatten()
    except ValueError:
        head_pos_z_series = np.repeat(99, nstep)
 
     ## Blink duration series
    try:
        blink_duration_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_average_blink_duration_u'][0][0]
        blink_duration_series = blink_duration_series.flatten()
    except ValueError:
        blink_duration_series = np.repeat(99, nstep)
 
    ## Left Gaze direction x series
    try:
        lgaze_dir_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_x'][0][0]
        lgaze_dir_x_series = lgaze_dir_x_series.flatten()
    except ValueError:
        lgaze_dir_x_series = np.repeat(99, nstep)
   
    ## Left Gaze direction y series
    try:
        lgaze_dir_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_y'][0][0]
        lgaze_dir_y_series = lgaze_dir_y_series.flatten()
    except ValueError:
        lgaze_dir_y_series = np.repeat(99, nstep)
   
     ## Left Gaze direction z series
    try:
        lgaze_dir_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_left_eye_gaze_direction_z'][0][0]
        lgaze_dir_z_series = lgaze_dir_z_series.flatten()
    except ValueError:
        lgaze_dir_z_series = np.repeat(99, nstep)
 
 
    ## Right Gaze direction x series
    try:
        rgaze_dir_x_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_x'][0][0]
        rgaze_dir_x_series = rgaze_dir_x_series.flatten()
    except ValueError:
        rgaze_dir_x_series = np.repeat(99, nstep)
   
    ## Right Gaze direction y series
    try:
        rgaze_dir_y_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_y'][0][0]
        rgaze_dir_y_series = rgaze_dir_y_series.flatten()
    except ValueError:
        rgaze_dir_y_series = np.repeat(99, nstep)
   
     ## Right Gaze direction z series
    try:
        rgaze_dir_z_series = data['elemDataI']['Output_FovioDMEResults_dme_core_gaze_right_eye_gaze_direction_z'][0][0]
        rgaze_dir_z_series = rgaze_dir_z_series.flatten()
    except ValueError:
        rgaze_dir_z_series = np.repeat(99, nstep)
 
      ## Gaze direction pitch
    try:
        gaze_pitch_series = data['elemDataI']['Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_pitch_d'][0][0]
        gaze_pitch_series = gaze_pitch_series.flatten()
    except ValueError:
        gaze_pitch_series = np.repeat(99, nstep)
 
      ## Gaze direction yaw
    try:
        gaze_yaw_series = data['elemDataI']['Output_FovioDMEResults_dgaze_unified_gaze_direction_deg_yaw_deg'][0][0]
        gaze_yaw_series = gaze_yaw_series.flatten()
    except ValueError:
        gaze_yaw_series = np.repeat(99, nstep)
 
      ## Left eyelid closed boolean
    try:
        right_closed_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_right_eyelid_closed'][0][0]
        right_closed_series = right_closed_series.flatten()
    except ValueError:
        right_closed_series = np.repeat(99, nstep)
 
      ## right eyelid closed boolean
    try:
        left_closed_series = data['elemDataI']['Output_FovioDMEResults_dme_core_eyelid_left_eyelid_closed'][0][0]
        left_closed_series = left_closed_series.flatten()
    except ValueError:
        left_closed_series = np.repeat(99, nstep)
 
    ## Vehicle Speed
    try:
        veh_speed_series = data['elemDataI']['VDS_Veh_Speed'][0][0]
        veh_speed_series = veh_speed_series.flatten()
    except ValueError:
        veh_speed_series = np.repeat(99, nstep)
 
 
    ## Brake Pedal Force
    try:
        brake_force_series = data['elemDataI']['CFS_Brake_Pedal_Force'][0][0]
        brake_force_series = brake_force_series.flatten()
    except ValueError:
        brake_force_series = np.repeat(99, nstep)
 
    ## Steering wheel angle
    try:
        steer_angle = data['elemDataI']['CFS_Steering_Wheel_Angle'][0][0]
        steer_angle = steer_angle.flatten()
    except ValueError:
        steer_angle = np.repeat(99, nstep)
 
   
    ## Steering wheel angle rate
    try:
        steer_angle_rate = data['elemDataI']['CFS_Steering_Wheel_Angle_Rate'][0][0]
        steer_angle_rate = steer_angle_rate.flatten()
    except ValueError:
        steer_angle_rate = np.repeat(99, nstep)
 
    ## Accelerator Pedal Position
    try:
        acc_pedal_series = data['elemDataI']['CFS_Accelerator_Pedal_Position'][0][0]
        acc_pedal_series = acc_pedal_series.flatten()
    except ValueError:
        acc_pedal_series = np.repeat(99, nstep)

    ## Left button presses
    try:
        left_button_series = data['elemDataI']['HI_LeftExtraButtons_QuarterCab'][0][0]
        left_button_series = left_button_series.flatten()
    except ValueError:
        left_button_series = np.repeat(99, nstep)

    ## Right button presses
    try:
        right_button_series = data['elemDataI']['HI_RightExtraButtons_QuarterCab'][0][0]
        right_button_series = right_button_series.flatten()
    except ValueError:
        right_button_series = np.repeat(99, nstep)

    ## Assemble DF w/ Subject and Drive meta data
    temp_df = pd.DataFrame({'Subject': np.repeat(reduced_disp['Subject'][i],  nstep),
                'Drive': np.repeat(reduced_disp['DriveN'][i],  nstep),
                'RPupil_Diameter': right_pupil_series,
                'LPupil_Diameter': left_pupil_series,
                'Lateral_Deviation': lat_pos_series,
                'Blink_Duration': blink_duration_series,
                'Left_Gaze_Dir_X': lgaze_dir_x_series,
                'Left_Gaze_Dir_Y': lgaze_dir_y_series,
                'Left_Gaze_Dir_Z': lgaze_dir_z_series,
                'Right_Gaze_Dir_X': rgaze_dir_x_series,
                'Right_Gaze_Dir_Y': rgaze_dir_y_series,
                'Right_Gaze_Dir_Z': rgaze_dir_z_series,
                'Gaze_Pitch': gaze_pitch_series,
                'Gaze_Yaw': gaze_yaw_series,
                'Left_Eyelid_Closed': left_closed_series,
                'Right_Eyelid_Closed': right_closed_series,
                'Vehicle_Speed': veh_speed_series,
                'Brake_Pedal_Force': brake_force_series,
                'Steering_Wheel_Angle': steer_angle,
                'Steering_Wheel_Angle_Rate': steer_angle_rate,
                'Accelerator_Pedal_Position': acc_pedal_series,
                'Head_Pos_X': head_pos_x_series,
                'Head_Pos_Y': head_pos_y_series,
                'Head_Pos_Z': head_pos_z_series,
                'Blink_Counter': blink_counter_series,
                'Blink_Frequency': blink_frequency_series,
                'Right_Button_Press': right_button_series,
                'Left_Button_Press': left_button_series,
                'DaqName': np.repeat(reduced_disp['MatName'][i][:-4], nstep)
 
    })
   
    ## Attach "Sample" for each 60 sec sample
    temp_df['Sample'] = 1 + temp_df.index // 3600
   
    ## Discard any samples that aren't length 3600
    temp_df = temp_df[temp_df.groupby('Sample')['Sample'].transform('size') == 3600]
   
    ## Append the temp df to the current final df
    final_df = pd.concat([final_df, temp_df], ignore_index=True)

final_df['Sample_ID'] = "ID_" + final_df['Subject'].astype(str) + "_" + final_df['Drive'].astype(str) + "_" + final_df['Sample'].astype(str)
final_df.to_csv('./data/non_aug_data.csv')

Skipping ./data/matfiles/NADS_IMPAIR_1A_20230821103336.mat: File not found. (2 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230817082432.mat: File not found. (6 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230818134403.mat: File not found. (15 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230823102131.mat: File not found. (18 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230824082406.mat: File not found. (27 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230830122641.mat: File not found. (34 / 195)
Successfully read ./data/matfiles/NADS_IMPAIR_3A_20230817111633.mat (7 / 195)
Skipping ./data/matfiles/NADS_IMPAIR_1A_20230825103904.mat: File not found. (39 / 195)
Successfully read ./data/matfiles/NADS_IMPAIR_3A_20230825083017.mat (37 / 195)
Successfully read ./data/matfiles/NADS_IMPAIR_3A_RS2_20230825083746.mat (38 / 195)
Successfully read ./data/matfiles/NADS_IMPAIR_3A_RS1_20230817112239.mat (8 / 195)
Successfully read ./data/matfiles/NADS_IMPAIR_2A_20230824110149.mat (28